# Product and Convolution of Gaussian Covariances

Multivariate Gaussians form a rich algebraic structure: the family is **closed under both product and convolution** (with appropriate normalization), and these two operations correspond to **dual covariance operations** with a natural geometric interpretation.

## Product of two Gaussians

The pointwise product of $\mathcal{N}(\mu_1, \Sigma_1)$ and $\mathcal{N}(\mu_2, \Sigma_2)$ (unnormalized) is again Gaussian with **precision matrix sum**:
$$
\mathcal{N}(\mu_1,\Sigma_1) \cdot \mathcal{N}(\mu_2, \Sigma_2) \propto \mathcal{N}(\mu^*, \Sigma^*), \qquad \Sigma^* = \left(\Sigma_1^{-1} + \Sigma_2^{-1}\right)^{-1}.
$$
The product covariance is **smaller** (more concentrated) than either factor — consistent with Bayes' rule: multiplying likelihoods combines information.

## Convolution of two Gaussians

The convolution $f_1 * f_2$ of two Gaussians has **covariance sum**:
$$
\mathcal{N}(\mu_1,\Sigma_1) * \mathcal{N}(\mu_2, \Sigma_2) = \mathcal{N}(\mu_1+\mu_2, \Sigma_1 + \Sigma_2).
$$
This is the **central limit theorem** in action: summing independent random variables adds their covariances.

## Duality via the Fourier transform

Product in the spatial domain corresponds to convolution in the Fourier domain and vice versa. For Gaussians:
$$
\widehat{\mathcal{N}(0,\Sigma)}(\xi) = \mathcal{N}(0, \Sigma^{-1}/(4\pi^2))(\xi),
$$
so the **Fourier dual of a covariance** $\Sigma$ is the precision $\Sigma^{-1}$. Product $\leftrightarrow$ precision sum, convolution $\leftrightarrow$ covariance sum.

## Ellipse representation

A 2D covariance matrix $\Sigma$ is visualized by the **standard deviation ellipse** $\{x : x^\top \Sigma^{-1} x = 1\}$ — the 1-$\sigma$ confidence ellipse of $\mathcal{N}(0,\Sigma)$. Product shrinks this ellipse; convolution expands it.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from ipywidgets import interact, FloatSlider

plt.rcParams['figure.dpi'] = 120

## Ellipse drawing and covariance operations

We implement a helper to draw the 1-$\sigma$ confidence ellipse, and functions computing the product and convolution covariances.

In [ ]:
def cov_ellipse(ax, Sigma, color='steelblue', alpha=0.3, n_std=1.0, label=None):
    """Draw n_std-sigma confidence ellipse of N(0, Sigma)."""
    vals, vecs = np.linalg.eigh(Sigma)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ell = Ellipse(xy=(0, 0), width=w, height=h, angle=theta,
                  facecolor=color, edgecolor=color,
                  alpha=alpha, linewidth=2, label=label)
    ax.add_patch(ell)
    # Outline only
    ell2 = Ellipse(xy=(0, 0), width=w, height=h, angle=theta,
                   facecolor='none', edgecolor=color, linewidth=2)
    ax.add_patch(ell2)

def rotation(t):
    return np.array([[np.cos(t), np.sin(t)], [-np.sin(t), np.cos(t)]])

def make_cov(angle, ratio):
    """2x2 covariance: rotation by angle, eccentricity ratio."""
    R = rotation(angle)
    return R @ np.diag([1.0, ratio]) @ R.T

def cov_product(S1, S2):
    """Product covariance: Sigma* = (Sigma1^{-1} + Sigma2^{-1})^{-1}"""
    return np.linalg.inv(np.linalg.inv(S1) + np.linalg.inv(S2))

def cov_convol(S1, S2):
    """Convolution covariance: Sigma* = Sigma1 + Sigma2"""
    return S1 + S2

print('Covariance operations ready.')

## Visualizing product and convolution

We fix a reference covariance $\Sigma_0$ (red) and sweep a second covariance $\Sigma_1(\theta)$ (blue) through different orientations. The product (green, smaller) and convolution (orange, larger) ellipses update accordingly.

In [ ]:
eta = 0.3**2
S0 = make_cov(np.pi/3, eta)  # fixed reference

n_frames = 6
angles = np.linspace(0, np.pi, n_frames, endpoint=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, angle in zip(axes.ravel(), angles):
    S1 = make_cov(angle, eta)
    S_prod = cov_product(S0, S1)
    S_conv = cov_convol(S0, S1)

    cov_ellipse(ax, S0, color='tomato', alpha=0.25, label='$\\Sigma_0$ (fixed)')
    cov_ellipse(ax, S1, color='royalblue', alpha=0.25, label=f'$\\Sigma_1(\\theta={angle/np.pi:.1f}\\pi)$')
    cov_ellipse(ax, S_prod, color='seagreen', alpha=0.4, label='product')
    cov_ellipse(ax, S_conv, color='darkorange', alpha=0.25, label='convolution')

    lim = 1.8
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_title(fr'$\theta = {angle/np.pi:.1f}\pi$', fontsize=9)
    ax.legend(fontsize=6, loc='upper right')

fig.suptitle('Product (green, smaller) and convolution (orange, larger) of Gaussian covariances',
             y=1.02)
plt.tight_layout()
plt.show()

## Algebraic properties

We verify numerically that:
1. **Commutativity**: $\Sigma_1 \boxplus \Sigma_2 = \Sigma_2 \boxplus \Sigma_1$ (both operations)
2. **Product is always smaller**: all eigenvalues of $\Sigma^* = (\Sigma_1^{-1} + \Sigma_2^{-1})^{-1}$ are $\leq$ those of $\Sigma_1$ and $\Sigma_2$
3. **Convolution is always larger**: eigenvalues of $\Sigma_1 + \Sigma_2 \geq \max(\Sigma_1, \Sigma_2)$ (elementwise on eigenvalues)

In [ ]:
n_angles = 100
angles_fine = np.linspace(0, 2*np.pi, n_angles)

# Track eigenvalues of product and convolution vs angle
eig_prod_max = []
eig_conv_min = []
eig_s0_max = np.linalg.eigvalsh(S0).max()
eig_s0_min = np.linalg.eigvalsh(S0).min()

for angle in angles_fine:
    S1 = make_cov(angle, eta)
    eig_s1_max = np.linalg.eigvalsh(S1).max()
    eig_s1_min = np.linalg.eigvalsh(S1).min()
    S_p = cov_product(S0, S1)
    S_c = cov_convol(S0, S1)
    eig_prod_max.append(np.linalg.eigvalsh(S_p).max())
    eig_conv_min.append(np.linalg.eigvalsh(S_c).min())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.plot(angles_fine/np.pi, eig_prod_max, 'seagreen', lw=2, label='$\\lambda_{\\max}(\\Sigma^*)$ product')
ax.axhline(eig_s0_max, color='tomato', lw=1.5, ls='--', label='$\\lambda_{\\max}(\\Sigma_0)$')
ax.set_xlabel('angle $\\theta/\\pi$'); ax.set_ylabel('eigenvalue')
ax.set_title('Product: $\\lambda_{\\max}(\\Sigma^*) \\leq \\lambda_{\\max}(\\Sigma_0)$')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(angles_fine/np.pi, eig_conv_min, 'darkorange', lw=2,
         label='$\\lambda_{\\min}(\\Sigma_1+\\Sigma_2)$')
ax2.axhline(eig_s0_min, color='tomato', lw=1.5, ls='--', label='$\\lambda_{\\min}(\\Sigma_0)$')
ax2.set_xlabel('angle $\\theta/\\pi$'); ax2.set_ylabel('eigenvalue')
ax2.set_title('Convolution: $\\lambda_{\\min}(\\Sigma_1+\\Sigma_2) \\geq \\lambda_{\\min}(\\Sigma_0)$')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interactive: sweep angle and eccentricity

Rotate $\Sigma_1$ and adjust its eccentricity to see how the product and convolution ellipses respond.

In [ ]:
def show_cov_ops(angle=0.5, eccentricity=0.1):
    S1 = make_cov(angle * np.pi, eccentricity**2)
    S_prod = cov_product(S0, S1)
    S_conv = cov_convol(S0, S1)

    fig, ax = plt.subplots(figsize=(6, 6))
    cov_ellipse(ax, S0, color='tomato', alpha=0.2, label='$\\Sigma_0$ (fixed, red)')
    cov_ellipse(ax, S1, color='royalblue', alpha=0.2, label='$\\Sigma_1$ (blue)')
    cov_ellipse(ax, S_prod, color='seagreen', alpha=0.4, label='product (green)')
    cov_ellipse(ax, S_conv, color='darkorange', alpha=0.2, label='convolution (orange)')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.legend(fontsize=9); ax.set_title('Gaussian covariance product and convolution')
    plt.tight_layout(); plt.show()

interact(show_cov_ops,
         angle=FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05,
                           description='angle/$\\pi$'),
         eccentricity=FloatSlider(value=0.3, min=0.05, max=0.9, step=0.05,
                                  description='eccentricity'));

## Bibliographical resources

- Petersen, K. B. and Pedersen, M. S. (2012). *The Matrix Cookbook*. Technical University of Denmark. Version 20121115.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. Section 2.3.
- Bhatia, R. (2007). *Positive Definite Matrices*. Princeton University Press.
- Villani, C. (2009). *Optimal Transport: Old and New*. Springer. Chapter 9 (Gaussian case).
- Tarantola, A. (2005). *Inverse Problem Theory and Methods for Model Parameter Estimation*. SIAM.